# Qwen2.5-0.5B Function-Calling SFT — Clean Training Notebook

This notebook is the focused SFT stage of the Tiny Function-Calling SLM project.

### Goal

Teach a small instruct model to produce reliable function/tool calls using **QLoRA + SFT**.

The notebook deliberately keeps the useful educational parts from the SmolLM2 notebook—especially parameter inspection and LoRA parameter understanding—while removing low-value manual training/debugging.

### Final flow

**Environment → Model → Parameter inspection → LoRA → Dataset → SFT → Save → Inference → Function-calling evaluation**

## 1. Environment

This cell is intentionally small. It verifies that the training environment is usable without turning the notebook into a dependency-debugging notebook.

In [1]:
import sys
import torch
import unsloth
import transformers
import trl
import peft

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Unsloth:", unsloth.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1006 13:14:30.150000 26588 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
Python: 3.12.13
PyTorch: 2.11.0+cu130
Unsloth: 2026.8.4
Transformers: 5.5.0
TRL: 0.24.0
PEFT: 0.20.0
CUDA available: True
CUDA: 13.0
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.0 GB


## 2. Training configuration

These are the few project-level choices we actually need to control.

- `MAX_SEQ_LENGTH`: maximum tokens used by the SFT examples.
- `LOAD_IN_4BIT`: loads the frozen base model in 4-bit precision.
- `SEED`: makes the experiment reproducible.

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_SEQ_LENGTH = 512
LOAD_IN_4BIT = True
SEED = 3407

print("Model:", MODEL_NAME)
print("Max sequence length:", MAX_SEQ_LENGTH)
print("4-bit:", LOAD_IN_4BIT)
print("Seed:", SEED)

Model: Qwen/Qwen2.5-0.5B-Instruct
Max sequence length: 512
4-bit: True
Seed: 3407


## 3. Load the base model

We use the **Instruct** checkpoint because our task is instruction → structured tool/function call generation.

Unsloth returns the model-matched tokenizer, which is the tokenizer we will use for SFT.

In [3]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
)

print("Model loaded.")
print("Chat template available:", tokenizer.chat_template is not None)
print("EOS:", tokenizer.eos_token, "| ID:", tokenizer.eos_token_id)
print("PAD:", tokenizer.pad_token, "| ID:", tokenizer.pad_token_id)

==((====))==  Unsloth 2026.8.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4050 Laptop GPU. Num GPUs = 1. Max memory: 5.997 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded.
Chat template available: True
EOS: <|im_end|> | ID: 151645
PAD: <|vision_pad|> | ID: 151654


## 4. Parameter inspection — keep this from the SmolLM2 notebook

This section is intentionally educational.

Before LoRA, the model contains **all base-model parameters**. With QLoRA, those base weights remain frozen while small trainable LoRA matrices are added.

The important distinction is:

- **Total parameters** = size of the original model.
- **Trainable parameters** = parameters actually updated during SFT.
- **Trainable %** = how much of the model we are learning.

This is worth keeping because it makes the PEFT idea concrete rather than treating LoRA as a black box.

In [4]:
# Full base-model parameter count
total_params_before = sum(p.numel() for p in model.parameters())

trainable_before = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params_before:,}")
print(f"Trainable parameters: {trainable_before:,}")
print(f"Trainable %:          {100 * trainable_before / total_params_before:.4f}%")

Total parameters:     342,185,856
Trainable parameters: 54,204,288
Trainable %:          15.8406%


### Parameter names and shapes

Looking at one or two parameter names is useful for understanding how the Transformer is organized and why names such as `q_proj`, `v_proj`, `gate_proj`, etc. are used as LoRA targets.

We do **not** print the entire parameter list; that creates noise without adding understanding.

In [5]:
for name, param in model.named_parameters():
    print(f"{name:70s} {tuple(param.shape)}")
    if "layers.0.self_attn.q_proj.weight" in name:
        break

model.embed_tokens.weight                                              (151936, 896)
model.layers.0.self_attn.q_proj.weight                                 (896, 896)


## 5. Apply LoRA / QLoRA

### What the main parameters mean

- `r`: LoRA rank. Higher rank gives the adapter more capacity and more trainable parameters.
- `lora_alpha`: scaling applied to the LoRA update.
- `lora_dropout`: dropout inside the adapter. We use `0.0` here.
- `target_modules`: Transformer projections where LoRA adapters are inserted.
- `bias="none"`: do not train bias parameters.
- `use_gradient_checkpointing="unsloth"`: reduces activation memory.

For a linear layer with input size `d_in` and output size `d_out`, LoRA replaces a full update with two low-rank matrices:

**A: `r × d_in`**  
**B: `d_out × r`**

So the trainable parameter count for that adapter is approximately:

**`r × (d_in + d_out)`**

This is the key reason LoRA can train a tiny fraction of the base model.

In [6]:
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

model.print_trainable_parameters()

Unsloth 2026.8.4 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


### Verify what became trainable

This is the useful part of the old SmolLM2 parameter inspection.

We only display a small sample and summarize the counts rather than dumping hundreds of adapter parameter names.

In [7]:
trainable_params = []
total_params_after = 0

for name, param in model.named_parameters():
    total_params_after += param.numel()
    if param.requires_grad:
        trainable_params.append((name, param.numel()))

print(f"Total parameters:     {total_params_after:,}")
print(f"Trainable parameters: {sum(n for _, n in trainable_params):,}")
print(f"Trainable %:          {100 * sum(n for _, n in trainable_params) / total_params_after:.4f}%")

print("\nSample trainable parameters:")
for name, n in trainable_params[:10]:
    print(f"{name:80s} {n:,}")

Total parameters:     350,984,064
Trainable parameters: 8,798,208
Trainable %:          2.5067%

Sample trainable parameters:
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight           14,336
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight           14,336
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight           14,336
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight           2,048
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight           14,336
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight           2,048
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight           14,336
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight           14,336
base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight              14,336
base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight              77,

In [8]:
model.train()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
            

## 6. Load the SFT dataset

The training data is already prepared as JSONL with a `text` field containing the chat-formatted training example.

We do not add another preprocessing framework here. Data preparation/quality belongs to the earlier workflow stage.

In [10]:
from typing import cast
from datasets import DatasetDict, load_dataset

DATA_DIR = "data/sft"

dataset = cast(
    DatasetDict,
    load_dataset(
        "json",
        data_files={
            "train": f"{DATA_DIR}/train.jsonl",
            "validation": f"{DATA_DIR}/val.jsonl",
        },
    ),
)

print(dataset)
print("Train columns:", dataset["train"].column_names)
print("Validation columns:", dataset["validation"].column_names)

DatasetDict({
    train: Dataset({
        features: ['id', 'messages', 'text'],
        num_rows: 48616
    })
    validation: Dataset({
        features: ['id', 'messages', 'text'],
        num_rows: 1169
    })
})
Train columns: ['id', 'messages', 'text']
Validation columns: ['id', 'messages', 'text']


## 7. Minimal dataset sanity checks

Before spending GPU time, verify that:

1. the expected `text` field exists;
2. there are no empty examples;
3. the training example actually contains the user/assistant structure.

In [11]:
assert "text" in dataset["train"].column_names
assert "text" in dataset["validation"].column_names

for split in ["train", "validation"]:
    rows = cast(list[dict[str, str]], dataset[split])
    empty = sum(not row["text"].strip() for row in rows)
    print(f"{split}: {len(dataset[split]):,} examples | empty: {empty}")
    assert empty == 0

print("\nSample training example:")
print(dataset["train"][0]["text"][:3000])

train: 48,616 examples | empty: 0
validation: 1,169 examples | empty: 0

Sample training example:
<|im_start|>system
You are a function-calling assistant.

Your task is to select the appropriate tool and provide the arguments required
to answer the user's request.

Return ONLY valid JSON.

The output must be a JSON array of tool calls in this format:

[
  {
    "name": "function_name",
    "arguments": {
      "argument_name": "value"
    }
  }
]

If none of the available tools can answer the request, return an empty array: []

Do not explain your answer.
Do not output markdown.
Do not invent tools.
<|im_end|>
<|im_start|>user
Available tools:

[
  {
    "name": "live_giveaways_by_type",
    "description": "Retrieve live giveaways from the GamerPower API based on the specified type.",
    "parameters": {
      "type": {
        "description": "The type of giveaways to retrieve (e.g., game, loot, beta).",
        "type": "str",
        "default": "game"
      }
    }
  }
]

User request

## 8. Token-length sanity check

This is the one preprocessing check worth keeping before training.

If many examples exceed `MAX_SEQ_LENGTH`, the model may lose part of the function call during truncation.

In [12]:
sample = dataset["train"][0]

encoded = tokenizer(
    sample["text"],
    truncation=False,
    return_tensors="pt",
)

print("Sample tokens:", encoded["input_ids"].shape[1])

# Small length audit
audit_examples = cast(
    list[dict[str, str]],
    dataset["train"].select(range(min(500, len(dataset["train"])))),
)

lengths = [
    len(tokenizer(example["text"], truncation=False)["input_ids"])
    for example in audit_examples
]

print("Checked:", len(lengths), "training examples")
print("Max length:", max(lengths))
print("Mean length:", round(sum(lengths) / len(lengths), 1))
print("Over MAX_SEQ_LENGTH:", sum(x > MAX_SEQ_LENGTH for x in lengths))

Sample tokens: 283
Checked: 500 training examples
Max length: 1504
Mean length: 586.7
Over MAX_SEQ_LENGTH: 273


## 9. SFT configuration

This is the actual training configuration.

We let **TRL's `SFTTrainer` handle the training loop**. We do not manually implement:

- `DataLoader`
- `collate_fn`
- forward pass
- backward pass
- optimizer step
- validation loop

Those manual cells were useful for learning in SmolLM2, but they are unnecessary here because the project is testing SFT, not reimplementing PyTorch.

In [14]:
from trl.trainer.sft_config import SFTConfig
from trl.trainer.sft_trainer import SFTTrainer

training_args = SFTConfig(
    output_dir="outputs/qwen25_05b_qlora",

    # Training
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    # Optimizer
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=0.05,

    # Precision
    fp16=False,
    bf16=True,

    # Logging / evaluation
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,

    # Checkpoints
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    # Reproducibility
    seed=SEED,

    # Dataset
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,

    # Performance
    optim="adamw_8bit",
    gradient_checkpointing=True,

    report_to="none",
)

## 10. Create the SFT trainer

`train_on_responses_only` is important for this task.

The model should learn the **assistant's function-call response**, rather than being rewarded for reproducing the user's instruction.

In [15]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args,
)

from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("SFTTrainer ready.")

Unsloth: Tokenizing ["text"]:   0%|          | 0/48616 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/1169 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/48616 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48616 [00:00<?, ? examples/s]

Unsloth: Removed 22254 out of 48616 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


Map:   0%|          | 0/1169 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1169 [00:00<?, ? examples/s]

Unsloth: Removed 507 out of 1169 samples from eval_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.
SFTTrainer ready.


## 11. Train

This is the only cell that should consume the bulk of the GPU time.

The important experiment is simply:

**baseline model → SFT model**

Do not add additional training machinery until the evaluation shows that it is actually needed.

In [ ]:
train_result = trainer.train()

print("Training complete.")
print(train_result)

## 12. Save the LoRA adapter

Save the adapter and tokenizer so the trained result can be evaluated or reloaded without repeating training.

In [ ]:
SAVE_PATH = "outputs/qwen25_05b_function_calling_lora"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Saved to:", SAVE_PATH)

## 13. Post-training parameter summary

This gives us the final answer to an important engineering question:

> How large is the base model, and how much did we actually train?

For a LoRA run, the adapter is tiny compared with the base model because the original weights remain frozen.

In [ ]:
trainable_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_after = sum(p.numel() for p in model.parameters())

print(f"Base/model parameters: {total_after:,}")
print(f"Trainable LoRA params: {trainable_after:,}")
print(f"Trainable percentage:  {100 * trainable_after / total_after:.4f}%")

## 14. Inference sanity check

Before calculating the complete evaluation metrics, generate one real response.

This verifies that the trained model can move through the same path it will use during evaluation:

**user request → chat template → model → assistant function call**

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

sample_text = dataset["validation"][0]["text"]

# Extract the user message from the stored chat-formatted example.
user_match = re.search(
    r"<\|im_start\|>user\s*(.*?)<\|im_end\|>",
    sample_text,
    flags=re.DOTALL,
)

if user_match:
    user_prompt = user_match.group(1).strip()
else:
    raise ValueError("Could not extract the user message from the training format.")

messages = [{"role": "user", "content": user_prompt}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=128,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

generated = tokenizer.decode(
    outputs[0][inputs.shape[1]:],
    skip_special_tokens=True,
)

print("USER:")
print(user_prompt)
print("\nMODEL:")
print(generated)

## 15. Function-calling evaluation

The final evaluation should use the project's function-calling metrics—not BLEU/ROUGE.

The core metrics are:

- **Parse Valid** — can the generated response be parsed?
- **Valid Call** — does it satisfy the expected function-call structure?
- **Tool EM** — exact tool/function match.
- **Function Accuracy** — correct function selected.
- **Argument Accuracy** — arguments match the gold call.

These are the metrics that answer whether SFT actually improved the model for our task.

Run the existing project evaluator here on the **same held-out evaluation set used for the baseline** and record the results beside the baseline.

In [ ]:
# Connect your existing function-calling evaluator here.
#
# Example expected result:
#
# sft_metrics = evaluate_function_calling(
#     model=model,
#     tokenizer=tokenizer,
#     eval_dataset=dataset["validation"],
# )
#
# print(sft_metrics)
#
# Expected keys:
# parse_valid
# valid_call
# tool_em
# function_accuracy
# argument_accuracy

print("Evaluation hook ready: run the project's existing function-calling evaluator here.")

## 16. Final comparison

The experiment is complete when the baseline and SFT model are compared on the same held-out examples.

| Metric | Base model | SFT model |
|---|---:|---:|
| Parse Valid | — | — |
| Valid Call | — | — |
| Tool EM | — | — |
| Function Accuracy | — | — |
| Argument Accuracy | — | — |

### Decision

If SFT improves function/tool selection and argument correctness without unacceptable formatting failures, the training stage has achieved its purpose.

Only after this comparison should we decide whether another training experiment is justified.